# Notebook 2 — Five-Minute AIS Interpolation Grid with Interactive Validation Map

Notebook ini:
1. membaca AIS bersih dari Notebook 1;
2. membentuk grid waktu 5 menit;
3. melakukan interpolasi posisi/kecepatan/arah di antara dua observasi AIS;
4. menggabungkan profil kapal;
5. menghasilkan output utama untuk Notebook 3;
6. menyediakan **peta interaktif validasi** agar hasil interpolasi bisa diperiksa secara visual.

Prinsip:
- hanya interpolasi;
- tidak ada carried-forward;
- tidak ada extrapolation;
- tidak ada label kualitas;
- `age_before_min`, `age_after_min`, dan `bracket_gap_min` tetap disimpan sebagai informasi;
- data kendaraan belum digabungkan di tahap ini.

Output utama:
- `02_vessel_interpolated_grid.csv`

Output validasi peta:
- `02_interpolated_full_validation_map.html`
- `02_interpolated_selected_validation_map.html`


In [18]:
#@title Setup and locate project root safely
from pathlib import Path
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Isi hanya bila nama/lokasi folder proyek Anda berbeda.
# Contoh:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"
PROJECT_ROOT_OVERRIDE = ""

def is_project_root(path):
    path = Path(path)
    return (
        path.is_dir()
        and (path / "stage_output").is_dir()
        and (path / "config").is_dir()
        and (path / "data_raw").is_dir()
    )

def locate_project_root():
    # 1. Lokasi manual selalu didahulukan.
    if PROJECT_ROOT_OVERRIDE:
        manual = Path(PROJECT_ROOT_OVERRIDE)
        if not is_project_root(manual):
            raise FileNotFoundError(
                "PROJECT_ROOT_OVERRIDE tidak menunjuk folder proyek yang valid: "
                f"{manual}"
            )
        return manual

    # 2. Mount Google Drive bila sedang berjalan di Colab.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except ImportError:
        pass

    # 3. Kandidat lokasi yang paling umum.
    direct_candidates = [
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
        Path.cwd(),
        Path.cwd().parent,
    ]

    for candidate in direct_candidates:
        if is_project_root(candidate):
            return candidate.resolve()

    # 4. Cari folder proyek di MyDrive dan Shared Drives.
    search_bases = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    for base in search_bases:
        if not base.exists():
            continue

        # os.walk dipangkas agar tidak mengobrak-abrik seluruh Drive tanpa alasan.
        for current, dirs, files in os.walk(base):
            current_path = Path(current)

            if current_path.name == "MFAR_Modular_Colab_Pipeline":
                if is_project_root(current_path):
                    return current_path.resolve()

            depth = len(current_path.relative_to(base).parts)
            if depth >= 5:
                dirs[:] = []

    raise FileNotFoundError(
        "Folder proyek MFAR tidak ditemukan. "
        "Pastikan folder/shortcut tersedia di Google Drive, atau isi "
        "PROJECT_ROOT_OVERRIDE dengan lokasi folder proyek yang benar."
    )

ROOT = locate_project_root()

STAGE1 = ROOT / "stage_output" / "stage_01"
STAGE2 = ROOT / "stage_output" / "stage_02"
CONFIG = ROOT / "config"

STAGE2.mkdir(parents=True, exist_ok=True)

INPUT_AIS = STAGE1 / "01_ais_clean.csv"
PROFILE_FILE = CONFIG / "vessel_profiles.csv"
BERTH_FILE = CONFIG / "terminal_berths.csv"

GRID_INTERVAL_MIN = 5
MAX_BRACKET_GAP_MIN = 20

print("Project root :", ROOT)
print("Input AIS    :", INPUT_AIS)
print("Output folder:", STAGE2)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root : /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline
Input AIS    : /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_01/01_ais_clean.csv
Output folder: /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_02


In [19]:
#@title Read Stage 1 AIS
ais = pd.read_csv(INPUT_AIS)

required = [
    "timestamp", "mmsi", "latitude",
    "longitude", "sog", "cog"
]

missing = [c for c in required if c not in ais.columns]

if missing:
    raise ValueError(
        "Kolom wajib dari Notebook 1 tidak ditemukan: "
        + ", ".join(missing)
    )

ais["timestamp"] = pd.to_datetime(
    ais["timestamp"],
    errors="coerce"
)

for col in [
    "mmsi", "latitude", "longitude",
    "sog", "cog"
]:
    ais[col] = pd.to_numeric(
        ais[col],
        errors="coerce"
    )

ais = (
    ais.dropna(subset=required)
    .sort_values(["mmsi", "timestamp"])
    .reset_index(drop=True)
)

print("Jumlah AIS bersih:", len(ais))
print("Jumlah MMSI:", ais["mmsi"].nunique())
print(
    "Periode:",
    ais["timestamp"].min(),
    "sampai",
    ais["timestamp"].max()
)

display(ais.head())


Jumlah AIS bersih: 32245
Jumlah MMSI: 4
Periode: 2026-01-01 06:20:21 sampai 2026-03-31 23:39:10


,_id,valid,error_mesg,aistype,channel,msglen,immsi,mmsi,class,nav_status,...,loc.type,loc.coordinates[0],loc.coordinates[1],original,port_origin,callsign,timestamp_raw,inside_corridor_bbox,rejection_reason,time_gap_min
0,6955c224abe4a1123f661f25,True,NaN,3,B,28,525002060,525002060,A,0.0,...,Point,102.136302,1.449843,"!AIVDM,1,1,,B,37lcUC0P007CRgb0m64P0?w>0S3i,0*0B",4338,NaN,2026-01-01T07:31:40.000Z,True,NaN,NaN
1,6955c710abe4a1123f6637b7,True,NaN,3,A,28,525002060,525002060,A,0.0,...,Point,102.136312,1.449842,"!AIVDM,1,1,,A,37lcUC0P007CRgn0m64BW?wN0S5i,0*07",4338,NaN,2026-01-01T07:31:50.000Z,True,NaN,0.166667
2,6955c530abe4a1123f662a91,True,NaN,1,B,28,525002060,525002060,A,0.0,...,Point,102.136292,1.449855,"!AIVDM,1,1,,B,17lcUC0P007CRgN0m66@0?v<2@3c,0*2F",4338,NaN,2026-01-01T07:39:09.000Z,True,NaN,7.316667
3,6955c710abe4a1123f6633b6,True,NaN,1,B,28,525002060,525002060,A,0.0,...,Point,102.136305,1.449855,"!AIVDM,1,1,,B,17lcUC0P007CRgf0m66@0?v>2@4A,0*20",4338,NaN,2026-01-01T07:52:09.000Z,True,NaN,13.000000
4,6955ca1cabe4a1123f666edd,True,NaN,1,B,28,525002060,525002060,A,0.0,...,Point,102.136307,1.449845,"!AIVDM,1,1,,B,17lcUC0P007CRgh0m64h0?v<2480,0*0F",4338,NaN,2026-01-01T08:00:07.000Z,True,NaN,7.966667


In [20]:
#@title Read or initialize vessel profiles
if PROFILE_FILE.exists():
    profiles = pd.read_csv(PROFILE_FILE)
else:
    profiles = pd.DataFrame()

if profiles.empty:
    profiles = pd.DataFrame({
        "mmsi": sorted(
            ais["mmsi"]
            .dropna()
            .astype("int64")
            .unique()
        ),
        "vessel_name": [
            f"MMSI_{x}"
            for x in sorted(
                ais["mmsi"]
                .dropna()
                .astype("int64")
                .unique()
            )
        ],
        "normal_sog_kn": [
            ais.loc[
                ais["mmsi"].astype("int64").eq(x),
                "sog"
            ].replace(0, np.nan).median()
            for x in sorted(
                ais["mmsi"]
                .dropna()
                .astype("int64")
                .unique()
            )
        ],
        "vehicle_capacity_ce": 30,
        "eta_reliability": 1.0,
        "berth_duration_reliability": 1.0,
        "turnaround_min": 40,
        "approach_allowance_min": 5
    })

    profiles.to_csv(
        PROFILE_FILE,
        index=False
    )

profiles["mmsi"] = pd.to_numeric(
    profiles["mmsi"],
    errors="coerce"
).astype("Int64")

display(profiles)


,mmsi,vessel_name,normal_sog_kn,vehicle_capacity_ce,eta_reliability,berth_duration_reliability,turnaround_min,approach_allowance_min
0,525002060,MMSI_525002060,6.7,30.0,1.0,1.0,40.0,5
1,525002121,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5
2,525003298,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5
3,525015047,MMSI_525015047,5.0,30.0,1.0,1.0,40.0,5


In [21]:
#@title Define interpolation logic
def circular_interpolate_deg(
    value_before,
    value_after,
    weight
):
    if pd.isna(value_before) or pd.isna(value_after):
        return np.nan

    start = np.deg2rad(value_before)
    end = np.deg2rad(value_after)

    delta = np.arctan2(
        np.sin(end - start),
        np.cos(end - start)
    )

    result = start + weight * delta

    return float(
        np.rad2deg(result) % 360
    )


def interpolate_one_vessel(
    vessel_df,
    interval_min=5,
    max_gap_min=20
):
    vessel_df = (
        vessel_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset=["timestamp"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    mmsi = int(vessel_df["mmsi"].iloc[0])

    grid_times = pd.date_range(
        vessel_df["timestamp"].min().ceil(
            f"{interval_min}min"
        ),
        vessel_df["timestamp"].max().floor(
            f"{interval_min}min"
        ),
        freq=f"{interval_min}min"
    )

    accepted_rows = []
    rejected_rows = []

    time_values = (
        vessel_df["timestamp"]
        .values
        .astype("datetime64[ns]")
    )

    for grid_time in grid_times:
        grid_np = np.datetime64(grid_time)

        after_index = np.searchsorted(
            time_values,
            grid_np,
            side="left"
        )

        if after_index >= len(vessel_df):
            rejected_rows.append({
                "grid_time": grid_time,
                "mmsi": mmsi,
                "rejection_reason": "no_after_observation"
            })
            continue

        if vessel_df.iloc[after_index]["timestamp"] == grid_time:
            before_index = after_index
        else:
            before_index = after_index - 1

        if before_index < 0:
            rejected_rows.append({
                "grid_time": grid_time,
                "mmsi": mmsi,
                "rejection_reason": "no_before_observation"
            })
            continue

        before = vessel_df.iloc[before_index]
        after = vessel_df.iloc[after_index]

        time_before = before["timestamp"]
        time_after = after["timestamp"]

        age_before_min = (
            grid_time - time_before
        ).total_seconds() / 60

        age_after_min = (
            time_after - grid_time
        ).total_seconds() / 60

        bracket_gap_min = (
            time_after - time_before
        ).total_seconds() / 60

        if bracket_gap_min < 0:
            rejected_rows.append({
                "grid_time": grid_time,
                "mmsi": mmsi,
                "rejection_reason": "negative_bracket_gap"
            })
            continue

        if bracket_gap_min > max_gap_min:
            rejected_rows.append({
                "grid_time": grid_time,
                "mmsi": mmsi,
                "source_time_before": time_before,
                "source_time_after": time_after,
                "age_before_min": age_before_min,
                "age_after_min": age_after_min,
                "bracket_gap_min": bracket_gap_min,
                "rejection_reason": "bracket_gap_too_large"
            })
            continue

        if bracket_gap_min == 0:
            weight = 0.0
        else:
            weight = age_before_min / bracket_gap_min

        weight = float(np.clip(weight, 0, 1))

        latitude = before["latitude"] + weight * (after["latitude"] - before["latitude"])
        longitude = before["longitude"] + weight * (after["longitude"] - before["longitude"])
        sog = before["sog"] + weight * (after["sog"] - before["sog"])
        cog = circular_interpolate_deg(before["cog"], after["cog"], weight)

        nav_status_before = before["nav_status"] if "nav_status" in before.index else np.nan
        nav_status_after = after["nav_status"] if "nav_status" in after.index else np.nan
        nav_status = nav_status_before if age_before_min <= age_after_min else nav_status_after

        accepted_rows.append({
            "grid_time": grid_time,
            "mmsi": mmsi,
            "source_time_before": time_before,
            "source_time_after": time_after,
            "age_before_min": age_before_min,
            "age_after_min": age_after_min,
            "bracket_gap_min": bracket_gap_min,
            "interpolation_weight": weight,
            "latitude": latitude,
            "longitude": longitude,
            "sog": sog,
            "cog": cog,
            "nav_status": nav_status
        })

    return pd.DataFrame(accepted_rows), pd.DataFrame(rejected_rows)


In [22]:
#@title Generate five-minute interpolated grid
accepted_parts = []
rejected_parts = []

for mmsi, vessel_df in ais.groupby("mmsi"):
    accepted_vessel, rejected_vessel = interpolate_one_vessel(
        vessel_df=vessel_df,
        interval_min=GRID_INTERVAL_MIN,
        max_gap_min=MAX_BRACKET_GAP_MIN
    )

    if not accepted_vessel.empty:
        accepted_parts.append(accepted_vessel)

    if not rejected_vessel.empty:
        rejected_parts.append(rejected_vessel)

interpolated = pd.concat(accepted_parts, ignore_index=True) if accepted_parts else pd.DataFrame()
rejected_grid = pd.concat(rejected_parts, ignore_index=True) if rejected_parts else pd.DataFrame()

interpolated = interpolated.sort_values(["grid_time", "mmsi"]).reset_index(drop=True)

print("Grid berhasil diinterpolasi:", len(interpolated))
print("Grid ditolak:", len(rejected_grid))
print("Jumlah waktu grid unik:", interpolated["grid_time"].nunique())
print("Jumlah MMSI:", interpolated["mmsi"].nunique())

display(interpolated.head(20))


Grid berhasil diinterpolasi: 40328
Grid ditolak: 61408
Jumlah waktu grid unik: 19469
Jumlah MMSI: 4


,grid_time,mmsi,source_time_before,source_time_after,age_before_min,age_after_min,bracket_gap_min,interpolation_weight,latitude,longitude,sog,cog,nav_status
0,2026-01-01 06:25:00,525002121,2026-01-01 06:20:33,2026-01-01 06:26:01,4.450000,1.016667,5.466667,0.814024,1.380135,102.148413,0.0,5.322866,0.0
1,2026-01-01 06:25:00,525003298,2026-01-01 06:23:32,2026-01-01 06:26:01,1.466667,1.016667,2.483333,0.590604,1.449362,102.138481,0.0,0.000000,8.0
2,2026-01-01 06:30:00,525002121,2026-01-01 06:26:01,2026-01-01 06:31:04,3.983333,1.066667,5.050000,0.788779,1.380150,102.148422,0.0,352.626403,0.0
3,2026-01-01 06:30:00,525003298,2026-01-01 06:26:01,2026-01-01 06:31:04,3.983333,1.066667,5.050000,0.788779,1.449357,102.138480,0.0,0.000000,8.0
4,2026-01-01 06:35:00,525002121,2026-01-01 06:31:04,2026-01-01 06:39:00,3.933333,4.000000,7.933333,0.495798,1.380152,102.148413,0.0,325.065546,0.0
5,2026-01-01 06:35:00,525003298,2026-01-01 06:31:04,2026-01-01 06:39:04,3.933333,4.066667,8.000000,0.491667,1.449355,102.138480,0.0,0.000000,8.0
6,2026-01-01 06:40:00,525002121,2026-01-01 06:39:00,2026-01-01 06:52:04,1.000000,12.066667,13.066667,0.076531,1.380146,102.148403,0.0,294.600510,0.0
7,2026-01-01 06:40:00,525003298,2026-01-01 06:39:04,2026-01-01 06:52:04,0.933333,12.066667,13.000000,0.071795,1.449355,102.138482,0.0,0.000000,8.0
8,2026-01-01 06:45:00,525002121,2026-01-01 06:39:00,2026-01-01 06:52:04,6.000000,7.066667,13.066667,0.459184,1.380134,102.148412,0.0,241.603061,0.0
9,2026-01-01 06:45:00,525003298,2026-01-01 06:39:04,2026-01-01 06:52:04,5.933333,7.066667,13.000000,0.456410,1.449356,102.138485,0.0,0.000000,8.0


In [23]:
#@title Merge vessel profiles
interpolated["mmsi"] = interpolated["mmsi"].astype("int64")

profiles_merge = profiles.copy()
profiles_merge["mmsi"] = profiles_merge["mmsi"].astype("int64")

output_grid = interpolated.merge(
    profiles_merge,
    on="mmsi",
    how="left",
    validate="many_to_one"
)

output_grid["date"] = output_grid["grid_time"].dt.strftime("%Y-%m-%d")
output_grid["time_of_day"] = output_grid["grid_time"].dt.strftime("%H:%M")

display(output_grid.head(20))


,grid_time,mmsi,source_time_before,source_time_after,age_before_min,age_after_min,bracket_gap_min,interpolation_weight,latitude,longitude,...,nav_status,vessel_name,normal_sog_kn,vehicle_capacity_ce,eta_reliability,berth_duration_reliability,turnaround_min,approach_allowance_min,date,time_of_day
0,2026-01-01 06:25:00,525002121,2026-01-01 06:20:33,2026-01-01 06:26:01,4.450000,1.016667,5.466667,0.814024,1.380135,102.148413,...,0.0,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5,2026-01-01,06:25
1,2026-01-01 06:25:00,525003298,2026-01-01 06:23:32,2026-01-01 06:26:01,1.466667,1.016667,2.483333,0.590604,1.449362,102.138481,...,8.0,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5,2026-01-01,06:25
2,2026-01-01 06:30:00,525002121,2026-01-01 06:26:01,2026-01-01 06:31:04,3.983333,1.066667,5.050000,0.788779,1.380150,102.148422,...,0.0,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5,2026-01-01,06:30
3,2026-01-01 06:30:00,525003298,2026-01-01 06:26:01,2026-01-01 06:31:04,3.983333,1.066667,5.050000,0.788779,1.449357,102.138480,...,8.0,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5,2026-01-01,06:30
4,2026-01-01 06:35:00,525002121,2026-01-01 06:31:04,2026-01-01 06:39:00,3.933333,4.000000,7.933333,0.495798,1.380152,102.148413,...,0.0,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5,2026-01-01,06:35
5,2026-01-01 06:35:00,525003298,2026-01-01 06:31:04,2026-01-01 06:39:04,3.933333,4.066667,8.000000,0.491667,1.449355,102.138480,...,8.0,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5,2026-01-01,06:35
6,2026-01-01 06:40:00,525002121,2026-01-01 06:39:00,2026-01-01 06:52:04,1.000000,12.066667,13.066667,0.076531,1.380146,102.148403,...,0.0,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5,2026-01-01,06:40
7,2026-01-01 06:40:00,525003298,2026-01-01 06:39:04,2026-01-01 06:52:04,0.933333,12.066667,13.000000,0.071795,1.449355,102.138482,...,8.0,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5,2026-01-01,06:40
8,2026-01-01 06:45:00,525002121,2026-01-01 06:39:00,2026-01-01 06:52:04,6.000000,7.066667,13.066667,0.459184,1.380134,102.148412,...,0.0,MMSI_525002121,7.0,30.0,1.0,1.0,40.0,5,2026-01-01,06:45
9,2026-01-01 06:45:00,525003298,2026-01-01 06:39:04,2026-01-01 06:52:04,5.933333,7.066667,13.000000,0.456410,1.449356,102.138485,...,8.0,MMSI_525003298,4.8,30.0,1.0,1.0,40.0,5,2026-01-01,06:45


In [24]:
#@title Build operational state required by Stage 3
# Stage 2 sekarang tidak hanya membuat grid, tetapi juga membentuk status operasi
# agar Stage 3 dapat langsung membaca 02_vessel_interpolated_grid.csv.

if not BERTH_FILE.exists():
    raise FileNotFoundError(f"Konfigurasi dermaga tidak ditemukan: {BERTH_FILE}")

berths = pd.read_csv(BERTH_FILE)
required_berth_cols = ["port_id", "berth_id", "latitude", "longitude"]
missing_berth_cols = [c for c in required_berth_cols if c not in berths.columns]
if missing_berth_cols:
    raise ValueError(
        "terminal_berths.csv tidak lengkap. Kolom hilang: "
        + ", ".join(missing_berth_cols)
    )

berths["port_id"] = berths["port_id"].astype(str).str.strip().str.upper()
berths["berth_id"] = berths["berth_id"].astype(str).str.strip()
berths["latitude"] = pd.to_numeric(berths["latitude"], errors="coerce")
berths["longitude"] = pd.to_numeric(berths["longitude"], errors="coerce")
berths = berths.dropna(subset=["latitude", "longitude"]).copy()

if "occupancy_radius_nm" not in berths.columns:
    berths["occupancy_radius_nm"] = 0.12
berths["occupancy_radius_nm"] = pd.to_numeric(
    berths["occupancy_radius_nm"], errors="coerce"
).fillna(0.12)


def haversine_nm(lat1, lon1, lat2, lon2):
    lat1 = np.radians(pd.to_numeric(lat1, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon1, errors="coerce"))
    lat2 = np.radians(pd.to_numeric(lat2, errors="coerce"))
    lon2 = np.radians(pd.to_numeric(lon2, errors="coerce"))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 3440.065 * 2.0 * np.arcsin(np.sqrt(a))


# Jarak setiap titik terhadap setiap dermaga.
distance_metadata = []
for _, berth in berths.iterrows():
    safe_id = (
        str(berth["berth_id"])
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )
    col = f"distance_to_{safe_id}_nm"
    output_grid[col] = haversine_nm(
        output_grid["latitude"],
        output_grid["longitude"],
        berth["latitude"],
        berth["longitude"],
    )
    distance_metadata.append({
        "column": col,
        "berth_id": berth["berth_id"],
        "port_id": berth["port_id"],
        "radius_nm": float(berth["occupancy_radius_nm"]),
    })

distance_cols = [x["column"] for x in distance_metadata]
if not distance_cols:
    raise ValueError("Tidak ada dermaga valid pada terminal_berths.csv.")

nearest_col = output_grid[distance_cols].idxmin(axis=1)
col_to_berth = {x["column"]: x["berth_id"] for x in distance_metadata}
col_to_port = {x["column"]: x["port_id"] for x in distance_metadata}
col_to_radius = {x["column"]: x["radius_nm"] for x in distance_metadata}

output_grid["nearest_berth_id"] = nearest_col.map(col_to_berth)
output_grid["nearest_port_id"] = nearest_col.map(col_to_port)
output_grid["nearest_berth_radius_nm"] = nearest_col.map(col_to_radius)
output_grid["nearest_distance_nm"] = output_grid[distance_cols].min(axis=1)

# Jarak terhadap pusat terminal untuk menentukan arah perjalanan.
port_centres = (
    berths.groupby("port_id")[["latitude", "longitude"]]
    .mean()
    .sort_index()
)
ports = list(port_centres.index)
if len(ports) != 2:
    raise ValueError(
        "Model koridor Stage 2 mengharapkan tepat dua terminal. "
        f"Ditemukan: {ports}"
    )

for port in ports:
    safe_port = str(port).lower().replace(" ", "_")
    output_grid[f"distance_to_{safe_port}_terminal_nm"] = haversine_nm(
        output_grid["latitude"],
        output_grid["longitude"],
        port_centres.loc[port, "latitude"],
        port_centres.loc[port, "longitude"],
    )

output_grid = output_grid.sort_values(["mmsi", "grid_time"]).reset_index(drop=True)

# Perubahan jarak ke masing-masing terminal per interval 5 menit.
for port in ports:
    safe_port = str(port).lower().replace(" ", "_")
    dcol = f"distance_to_{safe_port}_terminal_nm"
    output_grid[f"delta_{safe_port}_distance_nm"] = (
        output_grid.groupby("mmsi")[dcol].diff()
    )

# Status sandar memakai radius masing-masing dermaga dan batas kecepatan.
STOPPED_SPEED_KN = 0.80
MANEUVER_SPEED_KN = 3.00
APPROACH_RADIUS_NM = 0.60
MOVEMENT_EPS_NM = 0.005

at_berth = (
    (output_grid["nearest_distance_nm"] <= output_grid["nearest_berth_radius_nm"])
    & (pd.to_numeric(output_grid["sog"], errors="coerce").fillna(0.0) <= STOPPED_SPEED_KN)
)

# Arah menuju terminal ditentukan dari terminal yang jaraknya menurun paling kuat.
port_a, port_b = ports
safe_a = str(port_a).lower().replace(" ", "_")
safe_b = str(port_b).lower().replace(" ", "_")
da = output_grid[f"delta_{safe_a}_distance_nm"]
db = output_grid[f"delta_{safe_b}_distance_nm"]

output_grid["destination"] = np.where(da < db, port_a, port_b)
output_grid["origin"] = np.where(output_grid["destination"].eq(port_a), port_b, port_a)

# Untuk baris awal tanpa delta, pilih terminal yang lebih dekat sebagai destination.
first_or_unknown = da.isna() | db.isna()
dist_a = output_grid[f"distance_to_{safe_a}_terminal_nm"]
dist_b = output_grid[f"distance_to_{safe_b}_terminal_nm"]
output_grid.loc[first_or_unknown, "destination"] = np.where(
    dist_a[first_or_unknown] <= dist_b[first_or_unknown], port_a, port_b
)
output_grid.loc[first_or_unknown, "origin"] = np.where(
    output_grid.loc[first_or_unknown, "destination"].eq(port_a), port_b, port_a
)

# Saat kapal sandar, terminal sandar adalah origin dan terminal seberang adalah destination.
output_grid.loc[at_berth, "origin"] = output_grid.loc[at_berth, "nearest_port_id"]
output_grid.loc[at_berth, "destination"] = np.where(
    output_grid.loc[at_berth, "nearest_port_id"].eq(port_a), port_b, port_a
)

output_grid["operational_status"] = "SAILING"
output_grid["current_berth_id"] = np.nan
output_grid.loc[at_berth, "current_berth_id"] = output_grid.loc[at_berth, "nearest_berth_id"]
output_grid.loc[at_berth, "operational_status"] = (
    "AT_BERTH_" + output_grid.loc[at_berth, "nearest_port_id"].astype(str)
)

# Near-terminal transition state.
near_terminal = output_grid["nearest_distance_nm"] <= APPROACH_RADIUS_NM
nearest_port_delta = np.where(
    output_grid["nearest_port_id"].eq(port_a), da, db
)
nearest_port_delta = pd.Series(nearest_port_delta, index=output_grid.index)

approaching = (
    near_terminal & ~at_berth
    & (nearest_port_delta < -MOVEMENT_EPS_NM)
    & (output_grid["sog"] <= MANEUVER_SPEED_KN)
)
departing = (
    near_terminal & ~at_berth
    & (nearest_port_delta > MOVEMENT_EPS_NM)
    & (output_grid["sog"] <= MANEUVER_SPEED_KN)
)

output_grid.loc[approaching, "destination"] = output_grid.loc[approaching, "nearest_port_id"]
output_grid.loc[approaching, "origin"] = np.where(
    output_grid.loc[approaching, "nearest_port_id"].eq(port_a), port_b, port_a
)
output_grid.loc[approaching, "operational_status"] = (
    "APPROACHING_" + output_grid.loc[approaching, "nearest_port_id"].astype(str)
)

output_grid.loc[departing, "origin"] = output_grid.loc[departing, "nearest_port_id"]
output_grid.loc[departing, "destination"] = np.where(
    output_grid.loc[departing, "nearest_port_id"].eq(port_a), port_b, port_a
)
output_grid.loc[departing, "operational_status"] = (
    "DEPARTING_" + output_grid.loc[departing, "nearest_port_id"].astype(str)
)

# Jarak ke terminal tujuan yang dipakai Stage 4 untuk ETA.
def destination_distance(row):
    destination = row["destination"]
    if destination not in port_centres.index:
        return np.nan
    return float(haversine_nm(
        row["latitude"],
        row["longitude"],
        port_centres.loc[destination, "latitude"],
        port_centres.loc[destination, "longitude"],
    ))

output_grid["distance_to_destination_terminal_nm"] = output_grid.apply(
    destination_distance, axis=1
)
output_grid["is_at_berth"] = at_berth.astype(bool)

# Audit status operasional.
operational_audit = pd.DataFrame({
    "check": [
        "missing_operational_status",
        "missing_origin",
        "missing_destination",
        "missing_destination_distance",
        "at_berth_without_berth_id",
        "origin_equals_destination",
    ],
    "failed_rows": [
        int(output_grid["operational_status"].isna().sum()),
        int(output_grid["origin"].isna().sum()),
        int(output_grid["destination"].isna().sum()),
        int(output_grid["distance_to_destination_terminal_nm"].isna().sum()),
        int((at_berth & output_grid["current_berth_id"].isna()).sum()),
        int(output_grid["origin"].eq(output_grid["destination"]).sum()),
    ],
})

status_summary = (
    output_grid["operational_status"]
    .value_counts(dropna=False)
    .rename_axis("operational_status")
    .reset_index(name="rows")
)

print("Kolom operasional untuk Stage 3 berhasil dibentuk.")
display(status_summary)
display(operational_audit)


Kolom operasional untuk Stage 3 berhasil dibentuk.


,operational_status,rows
0,SAILING,15938
1,AT_BERTH_BENGKALIS,11350
2,AT_BERTH_PAKNING,8820
3,APPROACHING_PAKNING,1632
4,APPROACHING_BENGKALIS,1119
5,DEPARTING_PAKNING,740
6,DEPARTING_BENGKALIS,729


,check,failed_rows
0,missing_operational_status,0
1,missing_origin,0
2,missing_destination,0
3,missing_destination_distance,0
4,at_berth_without_berth_id,0
5,origin_equals_destination,0


In [25]:
#@title Validate interpolation output
audit_checks = pd.DataFrame({
    "check": [
        "grid_time_on_5_minute",
        "source_before_not_after_grid",
        "source_after_not_before_grid",
        "non_negative_age_before",
        "non_negative_age_after",
        "bracket_gap_within_limit",
        "weight_between_zero_and_one",
        "duplicate_mmsi_grid_time",
        "missing_latitude",
        "missing_longitude",
        "missing_sog"
    ],
    "failed_rows": [
        int(((output_grid["grid_time"].dt.minute % GRID_INTERVAL_MIN).ne(0)).sum()),
        int((output_grid["source_time_before"] > output_grid["grid_time"]).sum()),
        int((output_grid["source_time_after"] < output_grid["grid_time"]).sum()),
        int(output_grid["age_before_min"].lt(0).sum()),
        int(output_grid["age_after_min"].lt(0).sum()),
        int(output_grid["bracket_gap_min"].gt(MAX_BRACKET_GAP_MIN).sum()),
        int((~output_grid["interpolation_weight"].between(0, 1)).sum()),
        int(output_grid.duplicated(["grid_time", "mmsi"]).sum()),
        int(output_grid["latitude"].isna().sum()),
        int(output_grid["longitude"].isna().sum()),
        int(output_grid["sog"].isna().sum())
    ]
})

display(audit_checks)

if audit_checks["failed_rows"].sum() > 0:
    raise ValueError("Validasi internal Notebook 2 gagal. Periksa tabel audit_checks.")


,check,failed_rows
0,grid_time_on_5_minute,0
1,source_before_not_after_grid,0
2,source_after_not_before_grid,0
3,non_negative_age_before,0
4,non_negative_age_after,0
5,bracket_gap_within_limit,0
6,weight_between_zero_and_one,0
7,duplicate_mmsi_grid_time,0
8,missing_latitude,0
9,missing_longitude,0


In [26]:
#@title Generate interpolation summaries
summary = pd.DataFrame({
    "metric": [
        "grid_interval_min",
        "maximum_bracket_gap_min",
        "input_ais_rows",
        "interpolated_grid_rows",
        "rejected_grid_rows",
        "vessel_count",
        "unique_grid_times",
        "start_grid_time",
        "end_grid_time",
        "median_age_before_min",
        "median_age_after_min",
        "median_bracket_gap_min",
        "p95_bracket_gap_min"
    ],
    "value": [
        GRID_INTERVAL_MIN,
        MAX_BRACKET_GAP_MIN,
        len(ais),
        len(output_grid),
        len(rejected_grid),
        output_grid["mmsi"].nunique(),
        output_grid["grid_time"].nunique(),
        output_grid["grid_time"].min(),
        output_grid["grid_time"].max(),
        output_grid["age_before_min"].median(),
        output_grid["age_after_min"].median(),
        output_grid["bracket_gap_min"].median(),
        output_grid["bracket_gap_min"].quantile(0.95)
    ]
})

by_vessel = (
    output_grid.groupby("mmsi")
    .agg(
        interpolated_rows=("grid_time", "size"),
        start_grid_time=("grid_time", "min"),
        end_grid_time=("grid_time", "max"),
        median_age_before_min=("age_before_min", "median"),
        median_age_after_min=("age_after_min", "median"),
        median_bracket_gap_min=("bracket_gap_min", "median"),
        p95_bracket_gap_min=("bracket_gap_min", lambda x: x.quantile(0.95)),
        minimum_sog=("sog", "min"),
        median_sog=("sog", "median"),
        maximum_sog=("sog", "max")
    )
    .reset_index()
)

gap_distribution = pd.DataFrame({
    "range": [
        "0–5 min",
        ">5–10 min",
        ">10–15 min",
        ">15–20 min"
    ],
    "rows": [
        output_grid["bracket_gap_min"].between(0, 5, inclusive="both").sum(),
        output_grid["bracket_gap_min"].gt(5).mul(output_grid["bracket_gap_min"].le(10)).sum(),
        output_grid["bracket_gap_min"].gt(10).mul(output_grid["bracket_gap_min"].le(15)).sum(),
        output_grid["bracket_gap_min"].gt(15).mul(output_grid["bracket_gap_min"].le(20)).sum()
    ]
})

display(summary)
display(by_vessel)
display(gap_distribution)


,metric,value
0,grid_interval_min,5
1,maximum_bracket_gap_min,20
2,input_ais_rows,32245
3,interpolated_grid_rows,40328
4,rejected_grid_rows,61408
5,vessel_count,4
6,unique_grid_times,19469
7,start_grid_time,2026-01-01 06:25:00
8,end_grid_time,2026-03-31 23:35:00
9,median_age_before_min,4.366667


,mmsi,interpolated_rows,start_grid_time,end_grid_time,median_age_before_min,median_age_after_min,median_bracket_gap_min,p95_bracket_gap_min,minimum_sog,median_sog,maximum_sog
0,525002060,13806,2026-01-01 07:35:00,2026-03-31 23:35:00,4.525000,3.133333,12.783333,13.083333,0.0,0.395785,10.205428
1,525002121,12215,2026-01-01 06:25:00,2026-03-31 22:00:00,4.233333,3.016667,8.350000,13.066667,0.0,0.152021,11.998129
2,525003298,3086,2026-01-01 06:25:00,2026-03-31 22:35:00,4.816667,4.158333,12.883333,13.066667,0.0,1.884848,7.599582
3,525015047,11221,2026-01-05 17:15:00,2026-03-30 23:10:00,4.216667,3.033333,9.016667,13.133333,0.0,0.548223,8.060784


,range,rows
0,0–5 min,7584
1,>5–10 min,11721
2,>10–15 min,20889
3,>15–20 min,134


In [27]:
#@title Create auditable validation sample
validation_sample = (
    output_grid[
        [
            "grid_time",
            "mmsi",
            "source_time_before",
            "source_time_after",
            "age_before_min",
            "age_after_min",
            "bracket_gap_min",
            "interpolation_weight",
            "latitude",
            "longitude",
            "sog",
            "cog"
        ]
    ]
    .groupby("mmsi", group_keys=False)
    .head(50)
    .reset_index(drop=True)
)

display(validation_sample.head(50))


,grid_time,mmsi,source_time_before,source_time_after,age_before_min,age_after_min,bracket_gap_min,interpolation_weight,latitude,longitude,sog,cog
0,2026-01-01 07:35:00,525002060,2026-01-01 07:31:50,2026-01-01 07:39:09,3.166667,4.150000,7.316667,0.432802,1.449847,102.136303,0.000000,37.888838
1,2026-01-01 07:40:00,525002060,2026-01-01 07:39:09,2026-01-01 07:52:09,0.850000,12.150000,13.000000,0.065385,1.449855,102.136293,0.000000,0.000000
2,2026-01-01 07:45:00,525002060,2026-01-01 07:39:09,2026-01-01 07:52:09,5.850000,7.150000,13.000000,0.450000,1.449855,102.136298,0.000000,0.000000
3,2026-01-01 07:50:00,525002060,2026-01-01 07:39:09,2026-01-01 07:52:09,10.850000,2.150000,13.000000,0.834615,1.449855,102.136303,0.000000,0.000000
4,2026-01-01 07:55:00,525002060,2026-01-01 07:52:09,2026-01-01 08:00:07,2.850000,5.116667,7.966667,0.357741,1.449851,102.136306,0.000000,0.000000
5,2026-01-01 08:00:00,525002060,2026-01-01 07:52:09,2026-01-01 08:00:07,7.850000,0.116667,7.966667,0.985356,1.449845,102.136307,0.000000,0.000000
6,2026-01-01 08:05:00,525002060,2026-01-01 08:00:19,2026-01-01 08:13:08,4.683333,8.133333,12.816667,0.365410,1.449844,102.136298,0.000000,325.943823
7,2026-01-01 08:10:00,525002060,2026-01-01 08:00:19,2026-01-01 08:13:08,9.683333,3.133333,12.816667,0.755527,1.449840,102.136293,0.000000,289.584915
8,2026-01-01 08:15:00,525002060,2026-01-01 08:13:08,2026-01-01 08:26:10,1.866667,11.166667,13.033333,0.143223,1.446016,102.136191,1.188747,254.024552
9,2026-01-01 08:20:00,525002060,2026-01-01 08:13:08,2026-01-01 08:26:10,6.866667,6.166667,13.033333,0.526854,1.435779,102.135927,4.372890,219.804604


In [28]:
#@title Save and verify Stage 2 tables
main_output = STAGE2 / "02_vessel_interpolated_grid.csv"

output_grid.to_csv(
    main_output,
    index=False
)

rejected_grid.to_csv(
    STAGE2 / "02_interpolation_rejected_grid.csv",
    index=False
)

summary.to_csv(
    STAGE2 / "02_interpolation_summary.csv",
    index=False
)

by_vessel.to_csv(
    STAGE2 / "02_interpolation_by_vessel.csv",
    index=False
)

gap_distribution.to_csv(
    STAGE2 / "02_interpolation_gap_distribution.csv",
    index=False
)

validation_sample.to_csv(
    STAGE2 / "02_grid_validation_sample.csv",
    index=False
)

audit_checks.to_csv(
    STAGE2 / "02_interpolation_audit_checks.csv",
    index=False
)

operational_audit.to_csv(
    STAGE2 / "02_operational_state_audit.csv",
    index=False
)

status_summary.to_csv(
    STAGE2 / "02_operational_status_summary.csv",
    index=False
)

# Verifikasi baca ulang. Menulis file tanpa membuktikan keberadaannya rupanya
# terlalu optimistis untuk sebuah pipeline penelitian.
if not main_output.exists():
    raise IOError(
        f"Gagal membuat output utama Stage 2: {main_output}"
    )

saved_check = pd.read_csv(
    main_output,
    nrows=5
)

missing_saved_columns = [
    c for c in [
        "grid_time", "mmsi", "latitude", "longitude",
        "sog", "cog", "age_before_min",
        "age_after_min", "bracket_gap_min",
        "operational_status", "current_berth_id",
        "origin", "destination",
        "distance_to_destination_terminal_nm"
    ]
    if c not in saved_check.columns
]

if missing_saved_columns:
    raise IOError(
        "File utama berhasil dibuat tetapi kolomnya tidak lengkap: "
        + ", ".join(missing_saved_columns)
    )

print("Output utama Stage 2 berhasil dibuat dan diverifikasi:")
print(main_output)
print("Ukuran file:", f"{main_output.stat().st_size:,}", "bytes")
print("Jumlah baris sumber:", len(output_grid))


Output utama Stage 2 berhasil dibuat dan diverifikasi:
/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_02/02_vessel_interpolated_grid.csv
Ukuran file: 21,193,285 bytes
Jumlah baris sumber: 40328


## Bagian validasi peta interaktif

Bagian ini menampilkan:
- **titik interpolasi** berwarna per MMSI;
- **lintasan interpolasi** berdasarkan urutan `grid_time`;
- opsional **overlay titik AIS asli** dari Notebook 1 sebagai pembanding.

Filter inspeksi:
- MMSI
- tanggal
- jam mulai
- jam akhir
- tampilkan titik interpolasi
- tampilkan lintasan interpolasi
- tampilkan AIS asli


In [29]:
#@title Prepare map dependencies
try:
    import folium
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "folium", "ipywidgets", "-q"
    ])
    import folium
    import ipywidgets as widgets
    from IPython.display import display, clear_output

if (CONFIG / "terminal_berths.csv").exists():
    berths = pd.read_csv(CONFIG / "terminal_berths.csv")
else:
    berths = pd.DataFrame([
        ["BENGKALIS", "BENGKALIS_BERTH_1", 1.449921, 102.136747, 0.12],
        ["BENGKALIS", "BENGKALIS_BERTH_2", 1.449491, 102.138242, 0.12],
        ["PAKNING", "PAKNING_BERTH_1", 1.380001, 102.148561, 0.12],
        ["PAKNING", "PAKNING_BERTH_2", 1.378790, 102.149836, 0.12]
    ], columns=[
        "port_id", "berth_id", "latitude",
        "longitude", "occupancy_radius_nm"
    ])

if (CONFIG / "pipeline_parameters.csv").exists():
    params = pd.read_csv(CONFIG / "pipeline_parameters.csv").set_index("parameter")["value"].to_dict()
else:
    params = {}

CORRIDOR_MIN_LAT = float(params.get("corridor_min_lat", 1.34))
CORRIDOR_MAX_LAT = float(params.get("corridor_max_lat", 1.49))
CORRIDOR_MIN_LON = float(params.get("corridor_min_lon", 102.10))
CORRIDOR_MAX_LON = float(params.get("corridor_max_lon", 102.18))

COLOR_POOL = [
    "blue", "orange", "purple", "cadetblue",
    "darkred", "darkgreen", "pink", "lightblue",
    "lightgreen", "black", "darkpurple", "gray"
]

UNIQUE_MMSI = sorted(output_grid["mmsi"].dropna().astype(int).unique().tolist())
COLOR_MAP = {mmsi: COLOR_POOL[i % len(COLOR_POOL)] for i, mmsi in enumerate(UNIQUE_MMSI)}

ais_compare = ais.copy()
ais_compare["mmsi"] = ais_compare["mmsi"].astype(int)


In [30]:
#@title Define interactive validation map
def build_interpolated_validation_map(
    grid_df,
    original_df=None,
    selected_mmsi=None,
    selected_date=None,
    start_hour=0,
    end_hour=24,
    include_interpolated_points=True,
    include_interpolated_lines=True,
    include_original_points=False,
    max_interpolated_points=5000,
    max_original_points=2500,
    output_name="02_interpolated_selected_validation_map.html"
):
    view = grid_df.copy()

    if selected_mmsi not in [None, "ALL"]:
        view = view[view["mmsi"].astype(int).eq(int(selected_mmsi))]

    if selected_date not in [None, "ALL"]:
        selected_date_dt = pd.to_datetime(selected_date).date()
        view = view[view["grid_time"].dt.date.eq(selected_date_dt)]

    view_hour = (
        view["grid_time"].dt.hour
        + view["grid_time"].dt.minute / 60
        + view["grid_time"].dt.second / 3600
    )

    view = view[view_hour.ge(float(start_hour)) & view_hour.lt(float(end_hour))].copy()

    if view.empty:
        raise ValueError("Tidak ada data interpolasi pada kombinasi filter tersebut.")

    compare = None
    if original_df is not None and include_original_points:
        compare = original_df.copy()

        if selected_mmsi not in [None, "ALL"]:
            compare = compare[compare["mmsi"].astype(int).eq(int(selected_mmsi))]

        if selected_date not in [None, "ALL"]:
            compare = compare[compare["timestamp"].dt.date.eq(selected_date_dt)]

        compare_hour = (
            compare["timestamp"].dt.hour
            + compare["timestamp"].dt.minute / 60
            + compare["timestamp"].dt.second / 3600
        )

        compare = compare[compare_hour.ge(float(start_hour)) & compare_hour.lt(float(end_hour))].copy()

    center_lat = view["latitude"].median()
    center_lon = view["longitude"].median()

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles="OpenStreetMap",
        control_scale=True
    )

    corridor_layer = folium.FeatureGroup(name="Research corridor", show=True)
    bbox = [
        [CORRIDOR_MIN_LAT, CORRIDOR_MIN_LON],
        [CORRIDOR_MIN_LAT, CORRIDOR_MAX_LON],
        [CORRIDOR_MAX_LAT, CORRIDOR_MAX_LON],
        [CORRIDOR_MAX_LAT, CORRIDOR_MIN_LON],
        [CORRIDOR_MIN_LAT, CORRIDOR_MIN_LON]
    ]

    folium.PolyLine(
        bbox,
        color="red",
        weight=3,
        dash_array="8,6",
        tooltip="Research corridor"
    ).add_to(corridor_layer)
    corridor_layer.add_to(m)

    berth_layer = folium.FeatureGroup(name="Berth locations", show=True)

    for _, b in berths.iterrows():
        folium.Marker(
            [b["latitude"], b["longitude"]],
            tooltip=str(b["berth_id"]),
            popup=f"<b>{b['berth_id']}</b><br>Port: {b['port_id']}",
            icon=folium.Icon(color="green", icon="anchor", prefix="fa")
        ).add_to(berth_layer)

        folium.Circle(
            [b["latitude"], b["longitude"]],
            radius=float(b.get("occupancy_radius_nm", 0.12)) * 1852,
            color="green",
            fill=True,
            fill_opacity=0.08
        ).add_to(berth_layer)

    berth_layer.add_to(m)

    for mmsi, vessel_df in view.groupby("mmsi"):
        mmsi = int(mmsi)
        color = COLOR_MAP[mmsi]

        vessel_layer = folium.FeatureGroup(name=f"MMSI {mmsi}", show=True)
        vessel_df = vessel_df.sort_values("grid_time")

        if include_interpolated_lines and len(vessel_df) >= 2:
            folium.PolyLine(
                vessel_df[["latitude", "longitude"]].values.tolist(),
                color=color,
                weight=3,
                opacity=0.8,
                tooltip=(
                    f"MMSI {mmsi}<br>"
                    f"Interpolated trajectory<br>"
                    f"{vessel_df['grid_time'].min()}<br>"
                    f"to {vessel_df['grid_time'].max()}"
                )
            ).add_to(vessel_layer)

        if include_interpolated_points:
            point_df = vessel_df.copy()

            if len(point_df) > max_interpolated_points:
                indexes = np.linspace(0, len(point_df) - 1, max_interpolated_points).astype(int)
                point_df = point_df.iloc[indexes]

            for _, r in point_df.iterrows():
                folium.CircleMarker(
                    [r["latitude"], r["longitude"]],
                    radius=4,
                    color=color,
                    weight=1,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.85,
                    tooltip=f"MMSI {mmsi} | {r['grid_time']}",
                    popup=folium.Popup(
                        (
                            f"<b>MMSI:</b> {mmsi}<br>"
                            f"<b>Grid time:</b> {r['grid_time']}<br>"
                            f"<b>Latitude:</b> {r['latitude']:.6f}<br>"
                            f"<b>Longitude:</b> {r['longitude']:.6f}<br>"
                            f"<b>SOG:</b> {r['sog']:.2f} kn<br>"
                            f"<b>COG:</b> {r['cog']:.2f}°<br>"
                            f"<b>Before:</b> {r['source_time_before']}<br>"
                            f"<b>After:</b> {r['source_time_after']}<br>"
                            f"<b>Age before:</b> {r['age_before_min']:.2f} min<br>"
                            f"<b>Age after:</b> {r['age_after_min']:.2f} min<br>"
                            f"<b>Bracket gap:</b> {r['bracket_gap_min']:.2f} min<br>"
                            f"<b>Weight:</b> {r['interpolation_weight']:.3f}"
                        ),
                        max_width=420
                    )
                ).add_to(vessel_layer)

        vessel_layer.add_to(m)

    if compare is not None and not compare.empty:
        original_layer = folium.FeatureGroup(name="Original AIS points", show=True)
        compare_df = compare.sort_values("timestamp")

        if len(compare_df) > max_original_points:
            indexes = np.linspace(0, len(compare_df) - 1, max_original_points).astype(int)
            compare_df = compare_df.iloc[indexes]

        for _, r in compare_df.iterrows():
            mmsi = int(r["mmsi"])

            folium.CircleMarker(
                [r["latitude"], r["longitude"]],
                radius=2,
                color="gray",
                weight=1,
                fill=True,
                fill_color="white",
                fill_opacity=0.9,
                tooltip=f"Original AIS | MMSI {mmsi} | {r['timestamp']}",
                popup=folium.Popup(
                    (
                        f"<b>Original AIS</b><br>"
                        f"<b>MMSI:</b> {mmsi}<br>"
                        f"<b>Timestamp:</b> {r['timestamp']}<br>"
                        f"<b>SOG:</b> {r['sog']:.2f} kn<br>"
                        f"<b>COG:</b> {r['cog']:.2f}°"
                    ),
                    max_width=350
                )
            ).add_to(original_layer)

        original_layer.add_to(m)

    legend_items = ""
    for mmsi in sorted(view["mmsi"].astype(int).unique()):
        color = COLOR_MAP[mmsi]
        count = int(view["mmsi"].astype(int).eq(mmsi).sum())

        legend_items += f'''
        <div style="display:flex;align-items:center;margin-bottom:5px;">
          <span style="width:13px;height:13px;border-radius:50%;background:{color};border:1px solid #333;margin-right:7px;"></span>
          MMSI {mmsi} ({count:,} grid points)
        </div>
        '''

    legend_html = f'''
    <div style="position:fixed;bottom:35px;right:20px;z-index:9999;background:rgba(255,255,255,0.95);border:2px solid #777;border-radius:6px;padding:10px;font-family:Arial;font-size:12px;max-height:300px;overflow-y:auto;">
      <b>Interpolated AIS legend</b>
      <div style="margin-top:8px;">{legend_items}</div>
      <hr>
      <div>Large coloured point: interpolated grid point</div>
      <div>Solid coloured line: interpolated trajectory</div>
      <div>Small gray point: original AIS point</div>
      <div>Red dashed line: research corridor</div>
      <div>Green marker/circle: berth and radius</div>
    </div>
    '''

    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl(collapsed=False, position="topright").add_to(m)

    lat_values = list(view["latitude"].dropna())
    lon_values = list(view["longitude"].dropna())

    if compare is not None and not compare.empty:
        lat_values += list(compare["latitude"].dropna())
        lon_values += list(compare["longitude"].dropna())

    if lat_values and lon_values:
        m.fit_bounds([
            [min(lat_values), min(lon_values)],
            [max(lat_values), max(lon_values)]
        ])

    output_path = STAGE2 / output_name
    m.save(str(output_path))

    print("Data interpolasi terpilih:", len(view), "titik")
    if compare is not None:
        print("Data AIS asli pembanding:", len(compare), "titik")
    print("MMSI:", sorted(view["mmsi"].astype(int).unique().tolist()))
    print("Periode:", view["grid_time"].min(), "sampai", view["grid_time"].max())
    print("Peta disimpan:", output_path)

    return m


In [31]:
#@title Generate full interpolated validation map
full_map = build_interpolated_validation_map(
    grid_df=output_grid,
    original_df=ais_compare,
    selected_mmsi="ALL",
    selected_date="ALL",
    start_hour=0,
    end_hour=24,
    include_interpolated_points=True,
    include_interpolated_lines=True,
    include_original_points=True,
    max_interpolated_points=1500,
    max_original_points=1200,
    output_name="02_interpolated_full_validation_map.html"
)

print(
    "Peta validasi penuh tersimpan:",
    STAGE2 / "02_interpolated_full_validation_map.html"
)

Data interpolasi terpilih: 40328 titik
Data AIS asli pembanding: 32245 titik
MMSI: [525002060, 525002121, 525003298, 525015047]
Periode: 2026-01-01 06:25:00 sampai 2026-03-31 23:35:00
Peta disimpan: /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_02/02_interpolated_full_validation_map.html
Peta validasi penuh tersimpan: /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_02/02_interpolated_full_validation_map.html


In [32]:
#@title Interactive MMSI, date, and hour validation map
available_dates = sorted(
    output_grid["grid_time"].dt.strftime("%Y-%m-%d").unique().tolist()
)

mmsi_selector = widgets.Dropdown(
    options=["ALL"] + [str(x) for x in UNIQUE_MMSI],
    value="ALL",
    description="MMSI:"
)

date_selector = widgets.Dropdown(
    options=["ALL"] + available_dates,
    value="ALL",
    description="Tanggal:"
)

start_hour_selector = widgets.IntSlider(
    value=0, min=0, max=23, step=1, description="Jam mulai:"
)

end_hour_selector = widgets.IntSlider(
    value=24, min=1, max=24, step=1, description="Jam akhir:"
)

show_grid_points_selector = widgets.Checkbox(
    value=True, description="Tampilkan titik interpolasi"
)

show_grid_lines_selector = widgets.Checkbox(
    value=True, description="Tampilkan lintasan interpolasi"
)

show_original_selector = widgets.Checkbox(
    value=True, description="Tampilkan AIS asli"
)

generate_button = widgets.Button(
    description="Buat peta validasi",
    button_style="primary"
)

map_output = widgets.Output()

def generate_selected_map(_):
    with map_output:
        clear_output(wait=True)

        if end_hour_selector.value <= start_hour_selector.value:
            print("Jam akhir harus lebih besar daripada jam mulai.")
            return

        try:
            selected_map = build_interpolated_validation_map(
                grid_df=output_grid,
                original_df=ais_compare,
                selected_mmsi=mmsi_selector.value,
                selected_date=date_selector.value,
                start_hour=start_hour_selector.value,
                end_hour=end_hour_selector.value,
                include_interpolated_points=show_grid_points_selector.value,
                include_interpolated_lines=show_grid_lines_selector.value,
                include_original_points=show_original_selector.value,
                max_interpolated_points=5000,
                max_original_points=3000,
                output_name="02_interpolated_selected_validation_map.html"
            )
            display(selected_map)

        except Exception as error:
            print("Peta tidak dapat dibuat:", error)

generate_button.on_click(generate_selected_map)

display(
    widgets.VBox([
        widgets.HTML("<b>Filter visual interpolasi AIS</b>"),
        mmsi_selector,
        date_selector,
        start_hour_selector,
        end_hour_selector,
        widgets.VBox([
            show_grid_points_selector,
            show_grid_lines_selector,
            show_original_selector
        ]),
        generate_button,
        map_output
    ])
)


In [33]:
#@title Completion report
print("Notebook 2 selesai.")
print("File output Stage 2:")

for file_name in sorted(STAGE2.glob("02_*")):
    print("-", file_name.name)


Notebook 2 selesai.
File output Stage 2:
- 02_grid_validation_sample.csv
- 02_interpolated_full_validation_map.html
- 02_interpolation_audit_checks.csv
- 02_interpolation_by_vessel.csv
- 02_interpolation_gap_distribution.csv
- 02_interpolation_rejected_grid.csv
- 02_interpolation_summary.csv
- 02_map_segment_audit.csv
- 02_operational_state_audit.csv
- 02_operational_status_summary.csv
- 02_validation_map_525002060_2026-03-18.html
- 02_validation_map_525002121_2026-01-05.html
- 02_validation_map_525003298_2026-03-24.html
- 02_validation_map_525015047_2026-02-01.html
- 02_vessel_interpolated_grid.csv


## Corrected segmented map validation

The original full-period map remains useful as a coverage map. The following
cell creates vessel-day validation maps and breaks lines at temporal, spatial,
and operational segment boundaries.

In [34]:
# ============================================================
# VALIDATION MAP — SEGMENTED, NOT ONE THREE-MONTH POLYLINE
# ============================================================
# This helper prevents visual jumps between days, voyages, and long time gaps.

def add_segmented_track(
    fmap,
    vessel_df,
    line_color="blue",
    line_weight=3,
    line_opacity=0.75,
    max_gap_min=20,
    max_jump_nm=1.5,
):
    from math import radians, sin, cos, asin, sqrt

    def hav_nm(a, b):
        lat1, lon1 = map(radians, a)
        lat2, lon2 = map(radians, b)
        dlat, dlon = lat2-lat1, lon2-lon1
        h = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
        return 3440.065 * 2 * asin(sqrt(max(0.0, h)))

    x = vessel_df.copy()
    x["grid_time"] = pd.to_datetime(x["grid_time"], errors="coerce")
    x = x.dropna(subset=["grid_time","latitude","longitude"]).sort_values("grid_time")

    x["time_gap_min"] = x["grid_time"].diff().dt.total_seconds().div(60)
    x["date_change"] = x["grid_time"].dt.date.ne(x["grid_time"].shift().dt.date)

    coords = list(zip(x["latitude"], x["longitude"]))
    jumps = [0.0]
    for i in range(1, len(coords)):
        jumps.append(hav_nm(coords[i-1], coords[i]))
    x["spatial_jump_nm"] = jumps

    # Break the line at day changes, long gaps, spatial jumps, or a new departure episode.
    status = x.get("operational_status", pd.Series("", index=x.index)).astype(str)
    prev_status = status.shift().fillna("")
    new_departure = status.str.startswith("DEPARTING") & ~prev_status.str.startswith("DEPARTING")

    x["segment_break"] = (
        x["date_change"]
        | x["time_gap_min"].gt(max_gap_min)
        | x["spatial_jump_nm"].gt(max_jump_nm)
        | new_departure
    )
    x["map_segment_id"] = x["segment_break"].cumsum()

    for _, seg in x.groupby("map_segment_id"):
        if len(seg) < 2:
            continue
        folium.PolyLine(
            seg[["latitude","longitude"]].values.tolist(),
            color=line_color,
            weight=line_weight,
            opacity=line_opacity,
        ).add_to(fmap)

    return x[[
        "grid_time","mmsi","map_segment_id","time_gap_min",
        "spatial_jump_nm","segment_break"
    ]]

# Create a compact validation map per vessel-day sample.
sample_days = (
    output_grid.assign(date=pd.to_datetime(output_grid["grid_time"]).dt.date)
    .groupby(["mmsi","date"]).size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .groupby("mmsi").head(1)
)

seg_audit = []
for _, pick in sample_days.iterrows():
    mmsi = pick["mmsi"]
    day = pick["date"]
    view = output_grid[
        (output_grid["mmsi"] == mmsi)
        & (pd.to_datetime(output_grid["grid_time"]).dt.date == day)
    ].copy()
    if view.empty:
        continue

    center = [view["latitude"].median(), view["longitude"].median()]
    fmap = folium.Map(location=center, zoom_start=12, control_scale=True)

    audit_part = add_segmented_track(fmap, view)
    seg_audit.append(audit_part)

    for _, r in view.iloc[::max(1, len(view)//150)].iterrows():
        folium.CircleMarker(
            [r["latitude"], r["longitude"]],
            radius=2,
            tooltip=f'{r["grid_time"]} | {r.get("operational_status","")}',
            fill=True,
        ).add_to(fmap)

    out = STAGE2 / f"02_validation_map_{mmsi}_{day}.html"
    fmap.save(str(out))

if seg_audit:
    pd.concat(seg_audit, ignore_index=True).to_csv(
        STAGE2/"02_map_segment_audit.csv", index=False
    )

print("Segmented validation maps saved in:", STAGE2)

Segmented validation maps saved in: /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/stage_output/stage_02
